# UniColor — Python quickstart

`unicolor` is a Cython extension over the UniColor C ABI, shipped as a
self-contained wheel: the native library travels inside the package, so
installing it needs neither Nim nor a compiler.

```
pip install unicolor
```

CI executes this notebook against the wheel the release actually publishes, so
an output below that stops matching fails the build.

In [1]:
import unicolor

unicolor.version(), unicolor.__version__

('1.1.0', '1.1.0')

## Colors

A `Color` is 4 float32 components plus a `SpaceTag`. Build one from a CSS
string or a space-specific factory; a bad build raises `ValueError` (the C
sentinel has `tag == unicolor.TAG_UNKNOWN`).

In [2]:
red = unicolor.parse('#ff0000')
red.tag, red.components, red.alpha, red.format_css()

(1, (1.0, 0.0, 0.0), 1.0, 'oklch(1.0000 0.0000 0.0000)')

In [3]:
unicolor.srgb(1.0, 0.0, 0.0).format_css(), unicolor.oklch(0.65, 0.18, 250.0).format_css()

('oklch(1.0000 0.0000 0.0000)', 'oklch(0.6500 0.1800 250.0000)')

In [4]:
# format_css(legacy=True) emits sRGB hex; default is OKLCH.
red.format_css(legacy=True)

'#ff0000'

## Conversion & gamut mapping

In [5]:
# convert() changes the space; gamut_map() maps an out-of-gamut color
# into a target space's realizable range.
(red.convert(unicolor.TAG_OKLCH).format_css(),
 unicolor.oklch(0.7, 0.3, 200.0).gamut_map(unicolor.TAG_SRGB).format_css(legacy=True))

('oklch(0.6280 0.2577 29.2339)', '#00b7c0')

## Contrast & distance

`contrast` defaults to the WCAG 2.2 ratio; named metrics like `apca` are
supported. `distance` is a perceptual ΔE under a named metric.

In [6]:
(unicolor.contrast(unicolor.parse('#000000'), unicolor.parse('#ffffff')),
 unicolor.contrast(unicolor.parse('#000000'), unicolor.parse('#ffffff'), metric='apca'))

(21.0, 106.04067321268862)

In [7]:
unicolor.distance(unicolor.parse('#ff0000'), unicolor.parse('#00ff00'), 'deltaE_ok')

0.5198128952063058

## Themes

A `Theme` is a 3-layer token tree (primitives / semantics / components). Build
it from `(name, color|None, alias|None)` tuples: primitives carry a color,
semantics and components carry an alias.

In [8]:
t = unicolor.theme(
    [('surface', unicolor.srgb(1.0, 1.0, 1.0), None),
     ('text', unicolor.srgb(0.0, 0.0, 0.0), None)],
    [('text.primary', None, 'text')],
)
t.count, t.has_role('text.primary')

(3, True)

In [9]:
print(t.resolve('text.primary').format_css())
print(t.export('css'))

oklch(0.0000 0.0000 0.0000)
/* Generated by UniColor 1.1.0 | format: css | target: oklch | schema: 0 */
:root {
  --surface: oklch(1.0000 0.0000 90.0828);
  --text: oklch(0.0000 0.0000 0.0000);
  --text.primary: oklch(0.0000 0.0000 0.0000);
}


## Palettes

A `Palette` is an immutable color set. `color_at(i)` indexes the discrete
structures; `sample(t)` reads an ordered ramp at `t in [0,1]`. Both raise
`ValueError` when the structure does not support the operation or the index is
out of range.

In [10]:
p = unicolor.palette(
    unicolor.PAL_TAG_ORDERED,
    [unicolor.parse('#ff0000'), unicolor.parse('#00ff00'), unicolor.parse('#0055ff')],
    unicolor.PAL_INTENT_SEQUENTIAL,
)
len(p), p.color_at(1).format_css(), p.sample(0.5).format_css()

(3, 'oklch(0.0000 1.0000 0.0000)', 'oklch(0.8664 0.2948 142.4953)')

## Import & validation

`import_theme` reconstructs a theme from a serialized source (JSON, CSS, ...);
`import_reported` returns the diagnostics without the target. `validate_theme`
/ `validate_palette` run every registered rule and return a scored report.

In [11]:
j = t.export('json')
t2 = unicolor.import_theme(j, 'json')
t2.count, t2.resolve('surface').tag

(3, 16)

In [12]:
rep = unicolor.validate_theme(t2)
rep.score, rep.worst, rep.rule_count

(100, 0, 1)

In [13]:
rep.rule(0)

Rule(name='contrast-text-primary', severity=0, metric=21.000000090837478, threshold=4.5, message='contrast-text-primary: pass AA, text.primary on surface contrast 21.000 >= 4.5')

## The C ABI underneath

The same entry points are reachable from anything that speaks C:

```c
const char *uc_version(void);
uc_color uc_parse(const char *s);
uc_color uc_convert(uc_color c, int target);
double uc_contrast(uc_color fg, uc_color bg);
uc_theme *uc_theme_make(uc_token *prim, size_t nprim, ...);
uc_validation *uc_validate_theme(uc_theme *t);
```

See `include/UniColor.h`, and the book for the full picture.